# Lab 3.2 &mdash; Real Embeddings & Cosine Similarity

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 2 &middot; Module 3 &mdash; Why Transformers?**

### What you'll do
- Turn words into real vectors with a sentence-embedding model
- Implement cosine similarity by hand in NumPy
- Find each word's nearest neighbour in embedding space

> **How this lab works (near-real):** these labs run **real Hugging Face Transformers** locally on CPU. Read the **Concept**, fill the real `___` blanks in **Build it** (real tokenizer / model / decoding calls), **Run it for real** to see the **actual model output**, note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real model output you can read. The genuine maths (attention, positional encoding, cosine) you still compute **by hand** &mdash; that is real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased` (tokenizer / fill-mask), `sentence-transformers/all-MiniLM-L6-v2` (embeddings), `prajjwal1/bert-tiny` (attention), `distilgpt2` (generation). First use downloads the weights (needs network), then they are cached. The hosted "GPT API" path uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 3 slides &mdash; Tokens & embeddings](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 3 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (used by the text-gen lab)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-03-02")
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
Each token becomes a vector &mdash; an **embedding** &mdash; learned so that words with similar
meaning point in similar directions. We get **real** embeddings from
`sentence-transformers/all-MiniLM-L6-v2` (a real model forward pass, mean-pooled to one vector per
word), then measure similarity ourselves with **cosine similarity** &mdash; the angle between two
vectors: `1.0` = same direction, `0` = unrelated. The `embed()` helper is real; the cosine maths is
yours (that is real mechanics worth knowing).

## Build it
`embed()` (given) is a real model. You implement **cosine** and the nearest-neighbour search.

In [ ]:
import numpy as np
_EMB = {}
def embed(texts):
    """Real sentence embeddings from all-MiniLM-L6-v2: model forward pass -> mean-pool -> unit vector."""
    import torch
    from transformers import AutoTokenizer, AutoModel
    if not _EMB:
        name = "sentence-transformers/all-MiniLM-L6-v2"
        _EMB["tok"] = AutoTokenizer.from_pretrained(name)
        _EMB["mdl"] = AutoModel.from_pretrained(name); _EMB["mdl"].eval()
    if isinstance(texts, str): texts = [texts]
    enc = _EMB["tok"](texts, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad(): out = _EMB["mdl"](**enc)
    mask = enc["attention_mask"].unsqueeze(-1).float()
    pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1)     # mean over REAL tokens
    return torch.nn.functional.normalize(pooled, dim=1).numpy()      # unit vectors -> dot == cosine

def cosine(a, b):
    return ___   # TODO: (a . b) / (||a|| * ||b||)   use np.dot, np.linalg.norm

def most_similar(word, words, E):
    i = words.index(word)
    best, best_sim = None, -2.0
    for j, other in enumerate(words):
        if j == i: continue
        s = ___   # TODO: cosine between embedding E[i] and E[j]
        if s > best_sim: best_sim, best = s, other
    return best

## Run it for real
Embed real words and inspect the geometry.

In [ ]:
try:
    WORDS = ["king", "queen", "man", "woman", "apple", "orange"]
    E = embed(WORDS)                       # real model -> one unit vector per word
    print("embedding dim:", E.shape[1])
    for w in WORDS:
        print(f"  {w:8s} nearest -> {most_similar(w, WORDS, E)}")
    print("cos(king,queen) =", round(cosine(E[0], E[1]), 3),
          "| cos(king,apple) =", round(cosine(E[0], E[4]), 3))
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## What to notice
- The vectors are **384-dimensional** &mdash; real learned features, not toy numbers.
- `king` and `queen` sit close (**high cosine**); `king` and `apple` sit far apart (**low cosine**). Meaning became geometry.
- Because `embed()` returns **unit** vectors, the dot product already *is* the cosine &mdash; that is why fast vector search uses dot products.

## Your turn (open task &mdash; no grader)
Add your own words to `WORDS` (try `paris`, `france`, `tokyo`, `japan`) and re-run. Do
countries cluster near their capitals? Try short sentences instead of single words &mdash; `embed`
handles those too. A "good" answer: nearest-neighbour matches your intuition about meaning, and you
can explain one case where it surprises you.

---
### Key takeaway
Embeddings turn meaning into geometry, and cosine measures it. This is the engine behind search, RAG and recommendations -- and Lab 3.7 turns it into real semantic search.

[&#8592; All Module 3 labs](./index.html) &nbsp;&middot;&nbsp; [Module 3 slides](../../presentation/day2-module3-why-transformers.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>